# Ghost in the Aether — Vote Ingester

Polls a public Microsoft Forms response feed through Microsoft Graph, deduplicates new answers, and writes audience guesses into the Eventhouse `Votes` table for the live dashboard.

**Presenter setup:** create the MS Form manually, copy the form ID into the parameters cell, then start this notebook when audience voting opens.

In [ ]:
%pip install azure-kusto-data azure-kusto-ingest requests -q

In [ ]:
# Parameters
FORM_ID = ""  # Required: paste the Microsoft Forms form ID here before running
POLL_INTERVAL_SECONDS = 5
MAX_RUNTIME_MINUTES = 60

KUSTO_CLUSTER_URI = "{{EVENTHOUSE_CLUSTER_URI}}"
KUSTO_DATABASE = "AetherEH"
VOTES_TABLE = "Votes"
KUSTO_TOKEN_RESOURCE = "https://kusto.kusto.windows.net"
GRAPH_TOKEN_RESOURCE = "https://graph.microsoft.com"

# Update these if your form labels differ.
SUSPECT_FIELD_HINTS = ["suspect", "who", "killer", "whodunit", "guess"]
CONFIDENCE_FIELD_HINTS = ["confidence", "certain"]
MOTIVE_FIELD_HINTS = ["motive", "why", "reason"]

# If your tenant exposes a different Forms response endpoint, override this template.
GRAPH_RESPONSES_URL_TEMPLATE = "https://graph.microsoft.com/v1.0/forms/{form_id}/responses"

## 1. Authenticate and connect

The notebook uses the Fabric notebook identity for Graph and Kusto tokens. No secrets or service principals are stored in the repo.

In [ ]:
import csv
import io
import json
import re
import time
from datetime import datetime, timezone

import requests
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
from azure.kusto.data.exceptions import KustoServiceError

if not FORM_ID:
    raise ValueError("FORM_ID is empty. Paste the Microsoft Forms form ID into the parameters cell before running.")

def graph_headers() -> dict:
    token = notebookutils.credentials.getToken(GRAPH_TOKEN_RESOURCE)
    return {"Authorization": f"Bearer {token}", "Accept": "application/json"}

def kusto_token_provider() -> str:
    return notebookutils.credentials.getToken(KUSTO_TOKEN_RESOURCE)

kcsb = KustoConnectionStringBuilder.with_aad_token_provider(KUSTO_CLUSTER_URI, kusto_token_provider)
kusto_client = KustoClient(kcsb)

print(f"✓ Connected to Kusto database target: {KUSTO_DATABASE}")
print(f"✓ Poll interval: {POLL_INTERVAL_SECONDS}s")

## 2. Ensure the `Votes` table exists

Deployment creates the table from `DatabaseSchema.kql`; this cell is idempotent so the notebook also works if run independently.

In [ ]:
create_votes_table = f"""
.create-merge table {VOTES_TABLE} (
    ResponseId: string,
    Timestamp: datetime,
    Suspect: string,
    Confidence: int,
    Motive: string
)
"""

kusto_client.execute_mgmt(KUSTO_DATABASE, create_votes_table)
print(f"✓ Ensured table exists: {VOTES_TABLE}")

## 3. Poll Microsoft Forms and ingest new votes

The loop tracks response IDs in memory, parses likely suspect/confidence/motive fields, and ingests only new responses.

In [ ]:
def normalize(value) -> str:
    return str(value or "").strip().lower()

def field_matches(label: str, hints: list[str]) -> bool:
    label_norm = normalize(label)
    return any(hint in label_norm for hint in hints)

def response_id(response: dict) -> str:
    return str(response.get("id") or response.get("responseId") or response.get("resourceData", {}).get("id") or "")

def response_timestamp(response: dict) -> str:
    raw = response.get("submittedDateTime") or response.get("createdDateTime") or response.get("lastModifiedDateTime")
    if raw:
        return raw
    return datetime.now(timezone.utc).isoformat()

def iter_answers(response: dict):
    answers = response.get("answers") or response.get("values") or response.get("fields") or []
    if isinstance(answers, dict):
        for key, value in answers.items():
            yield str(key), value
        return
    for answer in answers:
        if isinstance(answer, dict):
            label = answer.get("questionName") or answer.get("displayName") or answer.get("name") or answer.get("questionId") or ""
            value = answer.get("answer") or answer.get("value") or answer.get("text") or answer.get("response") or ""
            yield str(label), value

def parse_vote(response: dict) -> dict:
    suspect = "Unknown"
    confidence = 0
    motive = ""
    fallback_values = []

    for label, value in iter_answers(response):
        text_value = str(value or "").strip()
        if not text_value:
            continue
        fallback_values.append(text_value)
        if field_matches(label, SUSPECT_FIELD_HINTS):
            suspect = text_value
        elif field_matches(label, CONFIDENCE_FIELD_HINTS):
            match = re.search(r"\d+", text_value)
            confidence = int(match.group(0)) if match else 0
        elif field_matches(label, MOTIVE_FIELD_HINTS):
            motive = text_value

    if suspect == "Unknown" and fallback_values:
        suspect = fallback_values[0]
    return {
        "ResponseId": response_id(response),
        "Timestamp": response_timestamp(response),
        "Suspect": suspect,
        "Confidence": confidence,
        "Motive": motive,
    }

def to_csv_line(vote: dict) -> str:
    buffer = io.StringIO()
    writer = csv.writer(buffer, lineterminator="")
    writer.writerow([vote["ResponseId"], vote["Timestamp"], vote["Suspect"], vote["Confidence"], vote["Motive"]])
    return buffer.getvalue()

def fetch_responses() -> list[dict]:
    url = GRAPH_RESPONSES_URL_TEMPLATE.format(form_id=FORM_ID)
    response = requests.get(url, headers=graph_headers(), timeout=30)
    if response.status_code >= 400:
        raise RuntimeError(f"Graph request failed: {response.status_code} {response.text[:500]}")
    payload = response.json()
    if isinstance(payload, dict):
        return payload.get("value") or payload.get("responses") or []
    return payload if isinstance(payload, list) else []

def ingest_votes(votes: list[dict]) -> None:
    if not votes:
        return
    csv_payload = "\n".join(to_csv_line(vote) for vote in votes)
    command = f".ingest inline into table {VOTES_TABLE} <|\n{csv_payload}"
    kusto_client.execute_mgmt(KUSTO_DATABASE, command)

print("✓ Helpers loaded")

In [ ]:
seen_response_ids = set()
deadline = time.time() + (MAX_RUNTIME_MINUTES * 60)
total_ingested = 0

print("Starting vote ingestion loop. Stop the cell manually when voting closes.")

while time.time() < deadline:
    try:
        responses = fetch_responses()
        new_votes = []
        for item in responses:
            rid = response_id(item)
            if not rid or rid in seen_response_ids:
                continue
            vote = parse_vote(item)
            seen_response_ids.add(rid)
            new_votes.append(vote)

        ingest_votes(new_votes)
        total_ingested += len(new_votes)
        print(f"{datetime.now(timezone.utc).isoformat()} | new={len(new_votes)} | total={total_ingested}")
    except KustoServiceError as exc:
        print(f"✗ Kusto ingestion failed: {exc}")
        raise
    except Exception as exc:
        print(f"✗ Polling failed: {exc}")
        raise

    time.sleep(POLL_INTERVAL_SECONDS)

print(f"✓ Vote ingestion finished. Total ingested: {total_ingested}")